# Battery sizing across sixteen households, three selectable price lists and one legacy contract

Perfect-foresight (MILP) annual electricity cost for **sixteen Fluvius households**, solved once
per **battery capacity** and once per **tariff**, over the **whole 2024 calendar year**
(35 136 fifteen-minute intervals, leap year included). Every solve uses the *continuous*
formulation (`use_discrete_actions=False`), so each point is the theoretical optimum for that
(household, tariff, capacity) — no controller can do better.

## The price lists

Every household owns a PV self-supply device, so all four lists are the
*samooskrba* (self-supply) variants and every solve runs under `pricing_scheme="si_samooskrba"`
— intra-interval netting of PV against load, with only the residual metered.

| Plan | `paket_id` | Status | Import price [EUR/kWh] | Export credit [EUR/kWh] | Monthly fee |
|---|---|---|---|---|---|
| GEN-I Dinamični | `GENI_SAMO_DINAMICNI` | selectable | SIPX + 0.01199 | SIPX − 0.01199, per interval | 1.97 |
| GEN-I Redni | `GENI_SAMO_REDNI` | selectable | 0.10290 flat (ET) | 0.05390 flat, per interval | 0.99 |
| GEN-I Aktivni | `GENI_SAMO_AKTIVNI` | selectable | 4 blocks, 0.04090 → 0.19290 | 4 blocks, 0.00190 → 0.14990 | 0.99 |
| GEN-I Redni + NET metering | `GENI_NETMETERING` | **legacy** | 0.12990 flat (ET) | **nothing per interval** — netted annually | 0.99 |

(Monthly fees are the `mesecno_nadomestilo_eko` rates: `eko_racun=True` is the pricing
default, so the dynamic list carries about 12 EUR/a more in fixed supplier fee than the other
three.) Beyond that, all four add the same network charge, levies and VAT on the metered
offtake and the same ratchet excess-power charge, so what separates them is the supplier's
energy price and the way exported energy is credited.

## NET metering is a legacy contract, not a baseline

Annual netting is **closed to new customers**. It only exists for households that already hold
that contract, which makes it the wrong thing to measure anything against: an option nobody can
buy cannot be the reference case, and "NET metering wins" is not an answer to "which list should
this household be on". Under `LEGACY_TARIFFS` it is therefore excluded from every winner, every
spread and every headline in section 8, which resolve over `CURRENT_TARIFF_ORDER` — Dinamični,
Redni, Aktivni — and it is reported alongside them instead, in grey on every chart, answering
two separate questions:

* what a household **still on** the contract pays today, and what it would give up by leaving;
* what it should size a battery for **after** the contract ends, which is a selectable list.

Read every NET-metering number as belonging to that second population only.

**NET metering needs an annual settlement, not an interval one.** `si_obracun._cena_oddaje`
returns `0.0` for `TipOdkupa.NET_METERING`, which is correct at the interval level — that
contract credits nothing as you export, it nets exports against imports once a year, on the
*supplier energy* component only (network charges and levies stay on gross metered offtake
under the 2024 network act). Priced interval-by-interval it would look absurdly bad and the
solver would curtail rather than export. `run_milp_benchmark` therefore takes
`annual_netting_rate_eur_per_kwh`, which adds

    credit = rate * min(total imported, total exported)

to the objective as a single bounded variable, and books it on the closing interval — see the
`Netting_Credit_EUR` column. `rate` is the list's own VAT-inclusive ET price, read straight
off `PAKETI["GENI_NETMETERING"]`.

## What is new relative to the single-household version

* Sixteen households instead of one, selected from the PV Fluvius families by a documented
  rule rather than by hand.
* Four price lists per household instead of one — three selectable, one legacy — with the
  NET-metering settlement above.
* **ROI**: capex, NPV, discounted ROI %, IRR and simple payback for every point on every curve.
* **Cycle count**: equivalent full cycles measured on the *stored* energy, and used to derive a
  cycle-limited service life that feeds back into the ROI.
* **Sizing against price**: section 8.4 sweeps the installed storage cost and reports the
  optimal capacity at each level, so the answer is not tied to one assumed price
  (the assumption is 250 EUR/kWh).
* An explicit whole-year audit (section 4) — the horizon really is every interval of 2024.

In [ ]:
### Imports
import json
import math
import sys
import textwrap
import time
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import pulp
from joblib import Parallel, delayed
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

import importlib
import Data_Loader as dl
import Environment as environment_module
import Horizon_Comparison as hc
import MILP_Benchmark as milp_module
import Pricing_Functions as pricing_module

dl = importlib.reload(dl)
environment_module = importlib.reload(environment_module)
hc = importlib.reload(hc)
milp_module = importlib.reload(milp_module)
pricing_module = importlib.reload(pricing_module)

HouseholdEnvironment = environment_module.HouseholdEnvironment
run_milp_benchmark = milp_module.run_milp_benchmark
PAKETI = pricing_module.PAKETI
DDV = pricing_module.DDV

print("Imports ready.")

## 1. Configuration

Everything the study depends on lives in this one cell.

In [ ]:
### Households -------------------------------------------------------------
# All four PV Fluvius families. The non-PV families are deliberately out: three
# of the four price lists (every samooskrba variant, NET metering included)
# require a self-supply device, so a household without PV cannot be quoted them.
PV_DATASETS = ["Fluvius","Fluvius_PV","Fluvius_HP","Fluvius_EV","Fluvius_HP_EV", "Fluvius_PV_EV", "Fluvius_PV_HP", "Fluvius_PV_HP_EV"]
N_HOUSEHOLDS = 16
SMP_COUNTRY_ID = "Slovenia"

PRICE_COLUMN = "SMP"
GENERATION_COLUMN = "Feed_In_Volume_kWh"
CONSUMPTION_COLUMN = "Consumption_Volume_kWh"

### Tariffs -----------------------------------------------------------------
# One entry per plan. `annual_netting` marks the price lists whose export credit
# is settled once a year rather than per interval (NET metering); the rate is
# read off the package itself so it tracks any price update in si_paketi.py.
#
# `legacy` marks a list that is closed to new contracts. NET metering is one:
# it is only available to households that already hold it, so it is NOT a
# baseline anyone can choose into and it never competes for "cheapest list" in
# section 8. It is carried through the whole study as a reference line — what a
# household still on that contract pays today, and what it gives up when the
# contract ends — and every winner, spread and headline is resolved over
# CURRENT_TARIFF_ORDER only.
TARIFFS = {
    "Dinamični":  {"paket_id": "GENI_SAMO_DINAMICNI", "scheme": "si_samooskrba", "annual_netting": False, "legacy": False},
    "Redni":      {"paket_id": "GENI_SAMO_REDNI",     "scheme": "si_samooskrba", "annual_netting": False, "legacy": False},
    "Aktivni":    {"paket_id": "GENI_SAMO_AKTIVNI",   "scheme": "si_samooskrba", "annual_netting": False, "legacy": False},
    "Redni + NM": {"paket_id": "GENI_NETMETERING",    "scheme": "si_samooskrba", "annual_netting": True,  "legacy": True},
}
TARIFF_ORDER = list(TARIFFS)
LEGACY_TARIFFS = [t for t, s in TARIFFS.items() if s["legacy"]]
CURRENT_TARIFF_ORDER = [t for t in TARIFF_ORDER if t not in LEGACY_TARIFFS]

PRICING_REFERENCE_YEAR = 2026      # dataset is 2024; SI tariff regime to price under
PEAK_RESET_MONTHS = 1           # repository default: ratchet peak never resets
CONTRACTED_POWER_KW = None         # None -> env default (historical naive peak / 1.5)

### Battery -----------------------------------------------------------------
# 0-40 kWh, deliberately dense below 6 kWh. The single-household notebook swept
# to 100 kWh because unconstrained SIPX arbitrage keeps paying up there; here the
# marginal value of capacity falls below its cost within the first few kWh (see
# section 9's marginal-value chart), so a grid that jumps 0 -> 2.5 -> 5 would put
# every optimum on a grid point and hide where the curve actually crosses.
BATTERY_SIZES_KWH = np.array(
    [0.0, 1.0, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 7.5, 10.0, 12.5, 15.0, 20.0, 25.0, 30.0, 40.0]
)

CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95

# Power rating scales with capacity (a 5 kWh pack does not get a 10 kW inverter),
# capped at a typical residential connection/inverter limit.
C_RATE = 0.5                       # kW per kWh of capacity
INVERTER_MAX_KW = 11.0

SOC_FRACTION = 0.5                 # start AND end each solve at this share of capacity

### Solver ------------------------------------------------------------------
# Taken from Horizon_Comparison rather than retyped, so this study's costs stay
# comparable with every other whole-period MILP in the repository and one edit
# moves all of them together.
SOLVER_GAP_REL = hc.FULL_PERIOD_GAP_REL          # 0.1 % MIP gap
SOLVER_TIME_LIMIT_S = hc.FULL_PERIOD_TIME_LIMIT_S
# CBC is single-threaded, so the sweep parallelizes across jobs instead. Each
# worker holds a full-year model (~250k variables); raise only if RAM allows.
N_JOBS = 4

### Economics ---------------------------------------------------------------
CAPEX_EUR_PER_KWH = 250.0          # installed, storage only — the assumed price
CAPEX_FIXED_EUR = 1000.0           # hybrid inverter / install, independent of size
# Two different jobs, so two different lists. The first is the handful of levels
# drawn as cost lines on the marginal-value chart, where more than a few become
# unreadable; the second is the grid the optimal-size table in 8.4 is swept over.
CAPEX_SCENARIOS_EUR_PER_KWH = [150.0, 250.0, 350.0, 450.0]
CAPEX_TABLE_EUR_PER_KWH = [100.0, 150.0, 200.0, 250.0, 300.0, 350.0, 400.0, 450.0, 500.0, 550.0]
BATTERY_CALENDAR_LIFE_Y = 12       # warranty band for residential Li-ion
BATTERY_CYCLE_LIMIT_EFC = 6000     # equivalent full cycles before end of life
DISCOUNT_RATE = 0.05

### Caching -----------------------------------------------------------------
RESULTS_DIR = Path("Results")
RESULTS_DIR.mkdir(exist_ok=True)
HOUSEHOLD_INDEX_CSV = RESULTS_DIR / "fluvius_pv_household_index.csv"
SWEEP_CACHE_CSV = RESULTS_DIR / "multiuser_battery_sizing.csv"

print(f"{len(TARIFFS)} tariffs x {N_HOUSEHOLDS} households x "
      f"{len(BATTERY_SIZES_KWH)} capacities = "
      f"{len(TARIFFS) * N_HOUSEHOLDS * len(BATTERY_SIZES_KWH)} full-year MILP solves")
print(f"Capacities: {BATTERY_SIZES_KWH}")
print(f"Selectable price lists: {', '.join(CURRENT_TARIFF_ORDER)}")
print(f"Legacy (reference only, closed to new contracts): {', '.join(LEGACY_TARIFFS)}")
print(f"Storage assumed at {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh + {CAPEX_FIXED_EUR:,.0f} EUR fixed")

## 2. Selecting the five households

Scans every household in the four PV families once, caches the summary to
`Results/fluvius_pv_household_index.csv`, and picks five by a rule rather than by hand:

1. **One per family** — the household whose PV-to-load ratio is closest to *that family's own
   median*, i.e. the typical Fluvius_PV / _PV_EV / _PV_HP / _PV_HP_EV household.
2. **One export-heavy household** — closest to the 90th percentile PV-to-load ratio across the
   pooled candidates. Everything that separates the four tariffs is how they price exported
   energy, so the sample needs at least one household that exports a lot.

A household qualifies only with a complete year (35 136 intervals), no missing values, nonzero
PV, and annual consumption in a plausible residential band.

In [ ]:
MIN_ANNUAL_CONSUMPTION_KWH = 1000.0
MAX_ANNUAL_CONSUMPTION_KWH = 20000.0
EXPECTED_INTERVALS = 35136          # 366 days x 96 intervals — 2024 is a leap year


def build_household_index(datasets, cache_path):
    '''One row per household in `datasets`: annual load, annual PV, data validity.'''
    if Path(cache_path).exists():
        idx = pd.read_csv(cache_path)
        print(f"Loaded cached index: {len(idx)} households from {cache_path}")
        return idx

    rows = []
    for dataset in datasets:
        dataset_dir = dl.INPUT_DATA_DIR / dataset
        for csv_path in sorted(dataset_dir.glob("*.csv")):
            hid = int(csv_path.stem.rsplit(" ", 1)[-1])
            raw = pd.read_csv(
                csv_path, usecols=[CONSUMPTION_COLUMN, GENERATION_COLUMN]
            )
            rows.append({
                "Dataset": dataset,
                "Household": hid,
                "Intervals": len(raw),
                "NaNs": int(raw.isna().sum().sum()),
                "Consumption_kWh": float(raw[CONSUMPTION_COLUMN].sum()),
                "Generation_kWh": float(raw[GENERATION_COLUMN].sum()),
            })
        print(f"  scanned {dataset}", flush=True)

    idx = pd.DataFrame(rows)
    idx.to_csv(cache_path, index=False)
    print(f"Scanned {len(idx)} households -> cached to {cache_path}")
    return idx


def qualify(idx):
    ok = (
        (idx["Intervals"] == EXPECTED_INTERVALS)
        & (idx["NaNs"] == 0)
        & (idx["Generation_kWh"] > 0)
        & (idx["Consumption_kWh"].between(MIN_ANNUAL_CONSUMPTION_KWH, MAX_ANNUAL_CONSUMPTION_KWH))
    )
    out = idx[ok].copy()
    out["PV_to_Load"] = out["Generation_kWh"] / out["Consumption_kWh"]
    return out.reset_index(drop=True)


def select_households(pool, n_total):
    '''One typical household per family, then export-heavy households to fill up.'''
    picks = []                                  # (pool index, role)
    for dataset, grp in pool.groupby("Dataset", sort=True):
        target = grp["PV_to_Load"].median()
        picks.append((int((grp["PV_to_Load"] - target).abs().idxmin()), f"typical {dataset}"))

    quantile = 0.90
    while len(picks) < n_total:
        target = pool["PV_to_Load"].quantile(quantile)
        remaining = pool.drop(index=[i for i, _ in picks])
        picks.append((int((remaining["PV_to_Load"] - target).abs().idxmin()),
                      f"export-heavy (p{int(round(quantile * 100))})"))
        quantile -= 0.05                        # next fill-up slot, if ever needed

    chosen = pool.loc[[i for i, _ in picks]].copy()
    chosen["Role"] = [role for _, role in picks]
    return chosen.sort_values(["Dataset", "Household"]).reset_index(drop=True)


household_index = build_household_index(PV_DATASETS, HOUSEHOLD_INDEX_CSV)
candidates = qualify(household_index)
print(f"{len(candidates)} of {len(household_index)} households qualify "
      f"(complete year, no NaNs, PV > 0, load in "
      f"{MIN_ANNUAL_CONSUMPTION_KWH:,.0f}-{MAX_ANNUAL_CONSUMPTION_KWH:,.0f} kWh/a)")
print(f"PV-to-load ratio across candidates: "
      f"p10 {candidates['PV_to_Load'].quantile(0.10):.2f} | "
      f"median {candidates['PV_to_Load'].median():.2f} | "
      f"p90 {candidates['PV_to_Load'].quantile(0.90):.2f}")

selected = select_households(candidates, N_HOUSEHOLDS)
selected["Label"] = selected["Dataset"].str.replace("Fluvius_", "", regex=False) + " #" + selected["Household"].astype(str)
HOUSEHOLD_KEYS = list(zip(selected["Dataset"], selected["Household"]))
HOUSEHOLD_LABELS = dict(zip(HOUSEHOLD_KEYS, selected["Label"]))

selected[["Label", "Dataset", "Household", "Role", "Consumption_kWh", "Generation_kWh", "PV_to_Load"]].round(2)

## 3. Load the profiles and the SMP price series

Same pipeline as the single-household notebook: profile from `Input data`, `SMP` overwritten
with the Slovenian series, EUR/MWh inputs auto-detected and converted to EUR/kWh.

In [ ]:
smp_country = dl.load_smp_data(SMP_COUNTRY_ID)


def load_priced_profile(dataset, household_id):
    '''Household profile with the SI SMP series written into PRICE_COLUMN (EUR/kWh).'''
    frame = dl.load_household_data(household_id, dataset=dataset)

    smp_aligned = smp_country.reindex(frame.index, method="ffill")
    smp_series = pd.to_numeric(smp_aligned[PRICE_COLUMN], errors="coerce").ffill().bfill()
    # Auto-detect legacy EUR/MWh inputs and convert to EUR/kWh.
    smp_scale = 1000.0 if float(smp_series.abs().quantile(0.95)) > 2.0 else 1.0
    frame[PRICE_COLUMN] = (smp_series / smp_scale).astype(float)
    return frame


HOUSEHOLD_FRAMES = {key: load_priced_profile(*key) for key in HOUSEHOLD_KEYS}

first_frame = HOUSEHOLD_FRAMES[HOUSEHOLD_KEYS[0]]
step_minutes = float(
    first_frame.index.to_series().sort_values().diff().dropna().dt.total_seconds().median() / 60.0
)
STEPS_PER_DAY = int(round(1440.0 / step_minutes))
HOURS_PER_STEP = 24.0 / STEPS_PER_DAY

for key in HOUSEHOLD_KEYS:
    f = HOUSEHOLD_FRAMES[key]
    print(f"{HOUSEHOLD_LABELS[key]:>16}: {len(f):,} x {step_minutes:.0f} min  "
          f"{f.index[0]:%Y-%m-%d} -> {f.index[-1]:%Y-%m-%d %H:%M}  "
          f"load {f[CONSUMPTION_COLUMN].sum():7,.0f} kWh/a  "
          f"PV {f[GENERATION_COLUMN].sum():7,.0f} kWh/a  "
          f"SMP {f[PRICE_COLUMN].min():+.3f} .. {f[PRICE_COLUMN].max():+.3f} EUR/kWh")

## 4. Whole-year audit

Three things had to be checked before trusting any of the numbers below. Two were wrong.

**1. The horizon dropped its last interval.** `run_milp_benchmark` takes its step count from
`env.episode_length` when `n_steps` is not given, and the single-household notebook passed
`episode_length = len(dataset) - 1`. That silently left 2024-12-31 23:45 unsolved. Here
`n_steps=len(frame)` is passed explicitly and asserted below: every run covers 35 136 of
35 136 intervals.

**2. The annualization factor shrank a leap year.** The old cell computed
`ANNUALIZE = 365 / (last - first).days`. For 2024 that is `365 / 365.99 = 0.9973` — it scaled
a *complete* calendar year down by 0.27 %, fixed monthly charges included. These profiles are
exactly one calendar year, so the raw sums already are the annual figures: `ANNUALIZE = 1.0`.
The factor is kept only so a genuinely partial profile would still be handled.

**3. The Fluvius timestamps are local Brussels time labelled `Z`.** This one is a property of
the input data, not of the notebook, and it is identical in *every* Fluvius file — plain,
_EV, _HP, _PV, all of them — so it affects every notebook in this repository that reads them:

* `2024-03-31 02:00–02:45` is **missing** (four intervals) — CET→CEST spring forward.
* `2024-10-27 02:00–02:45` appears **twice** (four duplicated stamps) — CEST→CET fall back.

The two cancel, which is why the row count is still exactly `366 x 96 = 35 136` and why the
energy totals are right. What is *not* right is the clock: because `Data_Loader` parses the
`...Z` suffix as UTC and `si_cas.v_lokalni_cas` then converts that "UTC" stamp to
Europe/Ljubljana, every interval is placed **1–2 hours later** than it really occurred
relative to the SI tariff clock and to the (genuinely UTC) SMP series. Tariff time blocks —
which is the whole substance of the *Aktivni* list — and the SIPX price alignment are shifted
by that much.

This is reported rather than repaired: fixing it means re-localizing the Fluvius inputs
repository-wide, which would move every published result in this project and is a decision to
take deliberately, not as a side effect of a sizing study. The comparison below is unaffected
in its *relative* conclusions — all four tariffs read the same shifted clock — but the
absolute bills carry the shift.

In [ ]:
def audit_frame(frame):
    diffs = frame.index.to_series().diff().dropna()
    expected = pd.Timedelta(minutes=step_minutes)
    irregular = diffs[diffs != expected]
    year = frame.index[0].year
    days_in_year = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    tz = frame.index.tz
    return {
        "Intervals": len(frame),
        "First": frame.index[0],
        "Last": frame.index[-1],
        "Span_days": len(frame) * HOURS_PER_STEP / 24.0,
        "Monotonic": bool(frame.index.is_monotonic_increasing),
        "Months": int(frame.index.month.nunique()),
        "NaNs": int(frame[[CONSUMPTION_COLUMN, GENERATION_COLUMN, PRICE_COLUMN]].isna().sum().sum()),
        # DST artefacts: repeated stamps in October, skipped intervals in March.
        "Duplicate_stamps": int(frame.index.duplicated().sum()),
        "Skipped_intervals": int(
            round((irregular[irregular > expected] - expected).sum() / expected)
        ),
        "Whole_calendar_year": bool(
            frame.index[0] == pd.Timestamp(f"{year}-01-01 00:00", tz=tz)
            and frame.index[-1] == pd.Timestamp(f"{year}-12-31 23:45", tz=tz)
            and len(frame) == days_in_year * STEPS_PER_DAY
        ),
    }


audit = pd.DataFrame(
    {HOUSEHOLD_LABELS[key]: audit_frame(HOUSEHOLD_FRAMES[key]) for key in HOUSEHOLD_KEYS}
).T

# Hard requirements for a full-year solve.
assert audit["Monotonic"].all(), "Timestamps are not sorted"
assert (audit["NaNs"] == 0).all(), "Missing values in load / PV / price"
assert (audit["Months"] == 12).all(), "A profile does not span all 12 months"
assert audit["Whole_calendar_year"].all(), "A profile is not one whole calendar year"
assert audit["Intervals"].nunique() == 1, "Profiles have different lengths"

# One complete calendar year -> the raw sums already are the annual figures.
ANNUALIZE = 1.0 if audit["Whole_calendar_year"].all() else 365.0 / float(audit["Span_days"].iloc[0])
N_STEPS_FULL_YEAR = int(audit["Intervals"].iloc[0])

old_annualize = 365.0 / ((audit["Last"].iloc[0] - audit["First"].iloc[0]).total_seconds() / 86400.0)
print(f"All {len(audit)} profiles: {N_STEPS_FULL_YEAR:,} intervals x {step_minutes:.0f} min = "
      f"{float(audit['Span_days'].iloc[0]):.0f} days, 12 months, sorted, no missing values.")
print(f"ANNUALIZE = {ANNUALIZE:.4f}   (the old 365/(last-first).days rule gave {old_annualize:.4f}, "
      f"i.e. {100 * (1 - old_annualize):.2f} % low on a leap year)")

dup = int(audit["Duplicate_stamps"].max())
skip = int(audit["Skipped_intervals"].max())
if dup or skip:
    print(f"\nDST artefacts (all profiles): {dup} duplicated stamps, {skip} skipped intervals — "
          f"{'they cancel, so the row count is exact' if dup == skip else 'THEY DO NOT CANCEL'}.")
    print("  The Fluvius stamps are Brussels local time written with a 'Z' suffix, so every "
          "interval sits 1-2 h off the SI tariff clock and the UTC SMP series. See section 4 text.")
audit

## 5. Environment, MILP and per-run metrics

### Equivalent full cycles — what the old formula counted

The trajectory columns named `Charge_kW` / `Discharge_kW` are **kWh per interval on the AC
side** (`P_ch`, `P_dis` in the MILP). The store itself moves

    stored in  = Charge_kW    * charge_efficiency
    stored out = Discharge_kW / discharge_efficiency

The single-household notebook used `sum(Discharge_kW) / capacity`, i.e. the energy that came
*out of the inverter*, which is `discharge_efficiency` (5 %) **below** the energy that actually
left the cells — it undercounts every cycle. The definition used here is the standard
throughput one, on stored energy:

    EFC = (stored in + stored out) / (2 * capacity)

With the terminal SOC pinned to the initial SOC the two halves are equal, so this also equals
`stored out / capacity`. Both numbers are reported (`Equivalent_Full_Cycles` and
`EFC_AC_Legacy`) so the difference is visible rather than assumed.

In [ ]:
SOLVER_KWARGS = dict(msg=False, gapRel=SOLVER_GAP_REL, timeLimit=SOLVER_TIME_LIMIT_S)


def netting_rate_eur_per_kwh(tariff_key):
    '''VAT-inclusive supplier energy rate a NET-metering list settles annually.'''
    spec = TARIFFS[tariff_key]
    if not spec["annual_netting"]:
        return None
    return float(PAKETI[spec["paket_id"]].et) * (1.0 + float(DDV))


def build_env(frame, capacity_kwh, tariff_key):
    '''Environment for one (household, capacity, tariff). Power scales with capacity.'''
    spec = TARIFFS[tariff_key]
    power_kw = min(C_RATE * float(capacity_kwh), INVERTER_MAX_KW)
    step_kwh = power_kw * HOURS_PER_STEP
    return HouseholdEnvironment(
        dataset=frame,
        price_column=PRICE_COLUMN,
        generation_column=GENERATION_COLUMN,
        consumption_column=CONSUMPTION_COLUMN,
        action_mode="continuous",
        allow_curtailment=True,
        reset_mode="deterministic",
        # Full year. `run_milp_benchmark` reads its step count off this when
        # n_steps is not given, and the usual len(frame) - 1 drops 31 Dec 23:45.
        episode_length=len(frame),
        steps_per_day=STEPS_PER_DAY,
        battery_capacity_kwh=float(capacity_kwh),
        charge_efficiency=CHARGE_EFFICIENCY,
        discharge_efficiency=DISCHARGE_EFFICIENCY,
        max_charge_kwh=step_kwh,
        max_discharge_kwh=step_kwh,
        pricing_scheme=spec["scheme"],
        pricing_reference_year=PRICING_REFERENCE_YEAR,
        pricing_options={"paket_id": spec["paket_id"]},
        contracted_power_kw=CONTRACTED_POWER_KW,
        peak_reset_months=PEAK_RESET_MONTHS,
    )


def summarize(df, capacity_kwh, env, tariff_key):
    '''Per-run metrics extracted from a solved full-year trajectory.'''
    net_grid_kwh = (
        df["Consumption"] + df["Charge_kW"] + df["Spill_kW"]
        - df["Generation"] - df["Discharge_kW"]
    )
    pv_surplus = np.maximum(df["Generation"] - df["Consumption"], 0.0)
    grid_charged = np.maximum(df["Charge_kW"] - pv_surplus, 0.0)

    # Energy moved through the STORE, not through the inverter.
    stored_in_kwh = float(df["Charge_kW"].sum()) * CHARGE_EFFICIENCY
    stored_out_kwh = float(df["Discharge_kW"].sum()) / DISCHARGE_EFFICIENCY
    efc = (stored_in_kwh + stored_out_kwh) / (2.0 * capacity_kwh) if capacity_kwh > 0 else 0.0

    return {
        "Tariff": tariff_key,
        "Capacity_kWh": float(capacity_kwh),
        "Power_kW": min(C_RATE * float(capacity_kwh), INVERTER_MAX_KW),
        "Cost_EUR": float(df["Cum_Cost"].iloc[-1]) * ANNUALIZE,
        "Netting_Credit_EUR": float(df["Netting_Credit_EUR"].sum()) * ANNUALIZE,
        "Charged_kWh": float(df["Charge_kW"].sum()) * ANNUALIZE,
        "Discharged_kWh": float(df["Discharge_kW"].sum()) * ANNUALIZE,
        "Stored_Out_kWh": stored_out_kwh * ANNUALIZE,
        "Grid_Charged_kWh": float(grid_charged.sum()) * ANNUALIZE,
        "Import_kWh": float(net_grid_kwh.clip(lower=0).sum()) * ANNUALIZE,
        "Export_kWh": float((-net_grid_kwh).clip(lower=0).sum()) * ANNUALIZE,
        "Curtailed_kWh": float(df["Spill_kW"].sum()) * ANNUALIZE,
        "Peak_Import_kW": float(net_grid_kwh.max()) / HOURS_PER_STEP,
        "Equivalent_Full_Cycles": efc * ANNUALIZE,
        "EFC_AC_Legacy": (
            float(df["Discharge_kW"].sum()) * ANNUALIZE / capacity_kwh if capacity_kwh > 0 else 0.0
        ),
        # Horizon provenance — proves every cached row really is a full year.
        "N_Steps": int(len(df)),
        "First_Step": df["Date"].iloc[0],
        "Last_Step": df["Date"].iloc[-1],
    }


def solve_one(frame, dataset, household_id, tariff_key, capacity_kwh):
    '''One full-year continuous MILP. Starts and ends at SOC_FRACTION of capacity,
    so no capacity is handed free energy at t=0.'''
    t0 = time.time()
    env = build_env(frame, capacity_kwh, tariff_key)
    soc_kwh = SOC_FRACTION * env.battery_capacity_kwh
    df = run_milp_benchmark(
        env,
        use_discrete_actions=False,      # continuous battery setpoint
        n_steps=len(frame),              # explicit: every interval of the year
        initial_soc_kwh=soc_kwh,
        final_soc_kwh=soc_kwh,
        solver=pulp.PULP_CBC_CMD(**SOLVER_KWARGS),
        annual_netting_rate_eur_per_kwh=netting_rate_eur_per_kwh(tariff_key),
        verbose=False,
    )
    row = summarize(df, capacity_kwh, env, tariff_key)
    row.update({
        "Dataset": dataset,
        "Household": int(household_id),
        "Contracted_Power_kW": float(list(env.contracted_power_kw.values())[0]),
        "Solve_s": time.time() - t0,
    })
    return row


for key in TARIFF_ORDER:
    rate = netting_rate_eur_per_kwh(key)
    print(f"{key:>12}: {TARIFFS[key]['paket_id']:<20} "
          f"annual netting {'-' if rate is None else f'{rate:.6f} EUR/kWh (VAT incl.)'}")

## 6. The sweep

`N_HOUSEHOLDS x len(TARIFFS) x len(BATTERY_SIZES_KWH)` full-year MILPs. A single solve takes
25–150 s (the dynamic list is the slow one — it is the only tariff with a genuine arbitrage
signal), so the sweep runs across `N_JOBS` processes and caches every finished row to
`Results/multiuser_battery_sizing.csv`. Re-running the cell only solves what is missing, so an
interrupted sweep resumes.

In [ ]:
%%time
JOB_KEYS = ["Dataset", "Household", "Tariff", "Capacity_kWh"]


def load_cache(path):
    if not Path(path).exists():
        return pd.DataFrame(columns=JOB_KEYS)
    cached = pd.read_csv(path)
    cached["Capacity_kWh"] = cached["Capacity_kWh"].astype(float)
    cached["Household"] = cached["Household"].astype(int)
    return cached


cache = load_cache(SWEEP_CACHE_CSV)
done = set(map(tuple, cache[JOB_KEYS].to_numpy())) if len(cache) else set()

jobs = [
    (dataset, hid, tariff, float(cap))
    for (dataset, hid), tariff, cap in product(HOUSEHOLD_KEYS, TARIFF_ORDER, BATTERY_SIZES_KWH)
    if (dataset, hid, tariff, float(cap)) not in done
]
print(f"{len(done)} runs cached, {len(jobs)} to solve on {N_JOBS} workers.")

if jobs:
    sweep_start = time.time()
    fresh = []
    # Unordered generator + a flush every FLUSH_EVERY rows: results land on disk
    # as they finish, so killing the kernel mid-sweep costs at most that many
    # solves rather than the whole run.
    FLUSH_EVERY = 5
    stream = Parallel(n_jobs=N_JOBS, verbose=10, return_as="generator_unordered")(
        delayed(solve_one)(HOUSEHOLD_FRAMES[(dataset, hid)], dataset, hid, tariff, cap)
        for dataset, hid, tariff, cap in jobs
    )
    for k, row in enumerate(stream, start=1):
        fresh.append(row)
        if k % FLUSH_EVERY == 0 or k == len(jobs):
            pd.concat([cache, pd.DataFrame(fresh)], ignore_index=True).to_csv(
                SWEEP_CACHE_CSV, index=False
            )
    # Round-trip so freshly solved rows and previously cached ones share dtypes
    # (Timestamps come back as strings; mixing the two breaks any later compare).
    cache = load_cache(SWEEP_CACHE_CSV)
    print(f"\nSolved {len(jobs)} runs in {(time.time() - sweep_start) / 60:.1f} min "
          f"-> cached to {SWEEP_CACHE_CSV}")

wanted = pd.DataFrame(
    [(d, h, t, float(c)) for (d, h), t, c in product(HOUSEHOLD_KEYS, TARIFF_ORDER, BATTERY_SIZES_KWH)],
    columns=JOB_KEYS,
)
df_runs = wanted.merge(cache, on=JOB_KEYS, how="left")
assert df_runs["Cost_EUR"].notna().all(), "Some runs are still missing — re-run this cell."

# Every cached row must be a whole-year horizon, not just the ones solved today.
assert (df_runs["N_Steps"] == N_STEPS_FULL_YEAR).all(), "A cached run does not cover the full year"
print(f"All {len(df_runs)} runs cover {N_STEPS_FULL_YEAR:,} intervals "
      f"({df_runs['First_Step'].min()} -> {df_runs['Last_Step'].max()}).")
print(f"Solve time: total {df_runs['Solve_s'].sum() / 3600:.2f} h, "
      f"median {df_runs['Solve_s'].median():.0f} s, max {df_runs['Solve_s'].max():.0f} s")

## 7. Savings, cycles and ROI

`Savings_EUR` is measured against the **0 kWh solve of the same household on the same tariff** —
same profile, same perfect foresight, no battery. Comparing across tariffs is done separately,
on the cost level, in section 8.

**Capex.** `capex(C) = CAPEX_EUR_PER_KWH * C + CAPEX_FIXED_EUR` for `C > 0`. The fixed term is
the hybrid inverter and the installation, which a 5 kWh pack pays as surely as a 30 kWh one —
leaving it out is what makes small batteries look artificially attractive.

**Service life is whichever runs out first.** A pack rated `BATTERY_CYCLE_LIMIT_EFC` equivalent
full cycles, cycled `EFC` times a year, is spent after `BATTERY_CYCLE_LIMIT_EFC / EFC` years:

    life = min(BATTERY_CALENDAR_LIFE_Y, BATTERY_CYCLE_LIMIT_EFC / EFC)

Under the dynamic tariff the MILP cycles hard enough that this bites, and it should: a battery
that earns its arbitrage revenue by wearing itself out in 7 years cannot be appraised on a
12-year annuity.

**The four ROI numbers**, all over that service life at `DISCOUNT_RATE`:

* `NPV_EUR` = `Savings * (1 - (1+r)^-n)/r - capex` — present value of the whole investment.
* `ROI_pct` = `NPV / capex * 100` — discounted return on the money put in. Positive = worth doing.
* `IRR_pct` — the discount rate at which NPV is zero; compare it against the cost of capital.
* `Payback_y` = `capex / Savings` — simple, undiscounted, for intuition only.

`Net_Annual_EUR` (`Savings - capex * a`, with `a` the capital recovery factor) is the same
decision expressed per year; its peak over capacity is the optimal size — *provided the peak
is interior*. A peak sitting on the smallest or largest capacity swept is the grid answering,
not the economics, and `Optimum_At_Grid_Edge` flags it.

**Two questions, kept apart.** `CAPEX_FIXED_EUR` is charged once whatever the size, so at
residential capacities it dominates: 1 000 EUR of install against a 2.5 kWh pack is more than
the storage itself. `Net_Annual_StorageOnly_EUR` re-runs the same arithmetic with only the
per-kWh cost, which answers "is storage worth its own price on this tariff?" separately from
"does the whole retrofit pay?". They do not have the same answer here, and the marginal-value
chart in section 9 is where the first one is read off.

In [ ]:
# Both factors use the closed form for EVERY rate except exactly zero, where it
# is 0/0. Short-circuiting on `rate <= 0` instead would return the undiscounted
# factor for negative rates too, which silently flattens the whole left half of
# the IRR search into a constant and makes the bisection return its lower bound.
def capital_recovery_factor(rate, years):
    '''Constant yearly payment that repays 1 EUR of capital over `years`.'''
    years = max(1.0, float(years))
    if rate == 0:
        return 1.0 / years
    return rate / (1.0 - (1.0 + rate) ** -years)


def present_value_factor(rate, years):
    '''Present value of 1 EUR received yearly for `years` years.'''
    years = max(1.0, float(years))
    if rate == 0:
        return years
    return (1.0 - (1.0 + rate) ** -years) / rate


def irr(capex, annual_savings, years, lo=-0.95, hi=5.0, tol=1e-9):
    '''Rate where the NPV of a level annuity equals capex. NaN when the capital
    is never repaid within the service life, even undiscounted.'''
    if capex <= 0 or annual_savings <= 0:
        return np.nan
    if annual_savings * max(1.0, years) < capex:
        return np.nan
    npv = lambda r: annual_savings * present_value_factor(r, years) - capex
    if npv(lo) < 0:
        return np.nan
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if npv(mid) > 0:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return 0.5 * (lo + hi)


# Self-check: 1000 EUR returning 200 EUR/a for 10 years is a 15.1 % IRR, and an
# investment that exactly returns its capital undiscounted is a 0 % one.
assert abs(irr(1000.0, 200.0, 10) - 0.15098) < 1e-4, "IRR solver is off"
assert abs(irr(1000.0, 100.0, 10)) < 1e-6, "IRR of a break-even annuity should be 0"
assert np.isnan(irr(1000.0, 50.0, 10)), "Never-repaid capital should be NaN"
assert abs(present_value_factor(0.05, 10) - 7.72173) < 1e-4
assert abs(capital_recovery_factor(0.05, 12) - 0.11283) < 1e-4


df = df_runs.copy()
df["Label"] = [HOUSEHOLD_LABELS[(d, h)] for d, h in zip(df["Dataset"], df["Household"])]
# Sorted here, not at the end: the marginal-value column below is a within-group
# diff over capacity and is meaningless on unsorted rows.
df = df.sort_values(["Dataset", "Household", "Tariff", "Capacity_kWh"]).reset_index(drop=True)

# Savings against this household's own no-battery cost ON THE SAME TARIFF.
baseline = (
    df.loc[df["Capacity_kWh"] == 0.0]
    .set_index(["Dataset", "Household", "Tariff"])["Cost_EUR"]
)
df["Cost_No_Battery_EUR"] = [
    baseline.loc[(d, h, t)] for d, h, t in zip(df["Dataset"], df["Household"], df["Tariff"])
]
df["Savings_EUR"] = df["Cost_No_Battery_EUR"] - df["Cost_EUR"]

# Capex, cycle-limited life, and the economics that follow from them.
df["Capex_EUR"] = np.where(
    df["Capacity_kWh"] > 0,
    df["Capacity_kWh"] * CAPEX_EUR_PER_KWH + CAPEX_FIXED_EUR,
    0.0,
)
df["Cycle_Life_y"] = (
    BATTERY_CYCLE_LIMIT_EFC / df["Equivalent_Full_Cycles"].where(df["Equivalent_Full_Cycles"] > 0)
).fillna(np.inf)
df["Service_Life_y"] = np.minimum(BATTERY_CALENDAR_LIFE_Y, df["Cycle_Life_y"])
df["Life_Limited_By"] = np.where(
    df["Cycle_Life_y"] < BATTERY_CALENDAR_LIFE_Y, "cycles", "calendar"
)
df["Lifetime_Cycles"] = df["Equivalent_Full_Cycles"] * df["Service_Life_y"]

crf = np.array([capital_recovery_factor(DISCOUNT_RATE, n) for n in df["Service_Life_y"]])
pvf = np.array([present_value_factor(DISCOUNT_RATE, n) for n in df["Service_Life_y"]])

df["Annuity_Factor"] = crf
df["Net_Annual_EUR"] = df["Savings_EUR"] - df["Capex_EUR"] * crf
df["NPV_EUR"] = df["Savings_EUR"] * pvf - df["Capex_EUR"]

# The same question with the one-off install cost taken out, so "is storage
# worth its own price?" can be answered separately from "does the whole package
# pay?". With a fixed term this large relative to a residential pack, the two
# answers are not the same, and conflating them hides which one is binding.
storage_capex = df["Capacity_kWh"] * CAPEX_EUR_PER_KWH
df["Net_Annual_StorageOnly_EUR"] = df["Savings_EUR"] - storage_capex * crf
df["Marginal_Value_EUR_per_kWh"] = (
    df.groupby(["Label", "Tariff"], observed=True)["Savings_EUR"].diff()
    / df.groupby(["Label", "Tariff"], observed=True)["Capacity_kWh"].diff()
)
df["ROI_pct"] = 100.0 * df["NPV_EUR"] / df["Capex_EUR"].where(df["Capex_EUR"] > 0)
df["IRR_pct"] = [
    100.0 * irr(c, s, n) if c > 0 else np.nan
    for c, s, n in zip(df["Capex_EUR"], df["Savings_EUR"], df["Service_Life_y"])
]
df["Payback_y"] = (
    df["Capex_EUR"].where(df["Capex_EUR"] > 0) / df["Savings_EUR"].where(df["Savings_EUR"] > 0)
)
df["Break_Even_Capex_EUR_kWh"] = (
    (df["Savings_EUR"] / crf - CAPEX_FIXED_EUR) / df["Capacity_kWh"].where(df["Capacity_kWh"] > 0)
)

LABEL_ORDER = [HOUSEHOLD_LABELS[k] for k in HOUSEHOLD_KEYS]

print(f"Capex model: {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh + {CAPEX_FIXED_EUR:,.0f} EUR fixed "
      f"-> {CAPEX_EUR_PER_KWH * 10 + CAPEX_FIXED_EUR:,.0f} EUR for a 10 kWh system")
print(f"Service life: min({BATTERY_CALENDAR_LIFE_Y} y calendar, "
      f"{BATTERY_CYCLE_LIMIT_EFC:,} EFC / cycles-per-year)")
print(f"  cycle-limited in {int((df['Life_Limited_By'] == 'cycles').sum())} of "
      f"{int((df['Capacity_kWh'] > 0).sum())} sized runs")

df[df["Capacity_kWh"] > 0].groupby("Tariff")[
    ["Equivalent_Full_Cycles", "EFC_AC_Legacy", "Service_Life_y", "Savings_EUR", "ROI_pct"]
].median().round(2).reindex(TARIFF_ORDER)

## 8. Comparison

Five questions, five tables:

1. **Which list, with no battery?** The plain contract choice, before any investment (8.1).
2. **Which battery size, per household per list?** Both the size and the money it makes (8.2).
3. **Which list, at each household's best battery size, and does the battery pay?** (8.3)
4. **How does the answer move with the price of storage?** The optimal capacity swept over
   installed cost, since 250 EUR/kWh is an assumption and not a measurement (8.4).
5. **How hard does each list work the pack?** Cycle counts, and the size of the correction
   to the old formula (8.5).

**Winners are resolved over the selectable lists only.** NET metering is legacy — closed to new
contracts — so it is not eligible to be "cheapest" or "best": it appears in its own columns as
the reference for households that already hold it, and as the delta they would face when it
ends. Anything labelled `legacy` in these tables answers a different question from the row it
sits next to.

Every cost here is solved to a `SOLVER_GAP_REL` relative MIP gap, so differences smaller than
that are below the resolution of the study. The `Cheapest` columns say `A = B (tie)` rather
than naming a winner whenever the top two land inside it.

In [ ]:
# Every cost carries a SOLVER_GAP_REL-sized uncertainty, so two tariffs closer
# than that are not actually distinguishable by this study. `resolve_winner`
# names the cheapest only when it wins by more than the gap.
def resolve_winner(row, columns, gap=SOLVER_GAP_REL):
    ranked = row[columns].astype(float).sort_values()
    best, second = ranked.index[0], ranked.index[1]
    tolerance = gap * abs(float(ranked.iloc[0]))
    if abs(float(ranked.iloc[1]) - float(ranked.iloc[0])) <= tolerance:
        return f"{best} = {second} (tie)"
    return best


### 8.1 No battery — the contract choice on its own
# `Cheapest` and `Spread_EUR` are resolved over CURRENT_TARIFF_ORDER only. NET
# metering is closed to new contracts, so naming it "cheapest" would answer a
# question nobody can act on. It keeps its own column, and the delta against the
# best selectable list is the thing that matters to a household still holding it:
# negative = still ahead on the legacy contract, positive = already behind.
no_batt = (
    df[df["Capacity_kWh"] == 0.0]
    .pivot(index="Label", columns="Tariff", values="Cost_EUR")
    .reindex(index=LABEL_ORDER, columns=TARIFF_ORDER)
)
no_batt["Cheapest"] = no_batt.apply(resolve_winner, axis=1, columns=CURRENT_TARIFF_ORDER)
no_batt["Cheapest_EUR"] = no_batt[CURRENT_TARIFF_ORDER].min(axis=1)
no_batt["Spread_EUR"] = (
    no_batt[CURRENT_TARIFF_ORDER].max(axis=1) - no_batt[CURRENT_TARIFF_ORDER].min(axis=1)
)
for legacy in LEGACY_TARIFFS:
    no_batt[f"{legacy} - best_EUR"] = no_batt[legacy] - no_batt["Cheapest_EUR"]

print(f"Annual cost with no battery [EUR/a]   "
      f"(ties called within the {SOLVER_GAP_REL:.1%} MIP gap)")
print(f"Cheapest is resolved over {', '.join(CURRENT_TARIFF_ORDER)}; "
      f"{', '.join(LEGACY_TARIFFS)} is legacy and shown for reference only.")
display(no_batt.round(2))

for legacy in LEGACY_TARIFFS:
    delta = no_batt[f"{legacy} - best_EUR"]
    print(f"{legacy}: cheaper than the best selectable list for "
          f"{int((delta < 0).sum())} of {len(delta)} households, by "
          f"{-delta[delta < 0].median() if (delta < 0).any() else 0:,.2f} EUR/a at the median; "
          f"more expensive for {int((delta >= 0).sum())}.")

In [ ]:
### 8.2 Best battery size per (household, tariff), by net annual benefit
sized = df[df["Capacity_kWh"] > 0].copy()
best_idx = sized.groupby(["Label", "Tariff"])["Net_Annual_EUR"].idxmax()
best = sized.loc[best_idx].copy()

SMALLEST_SIZED = float(BATTERY_SIZES_KWH[BATTERY_SIZES_KWH > 0].min())
LARGEST_SIZED = float(BATTERY_SIZES_KWH.max())
# A peak that sits on the first or last swept capacity is not an interior optimum
# — the curve is still moving in one direction and the grid, not the economics,
# is picking the answer. Flagged rather than reported as an optimum.
best["Optimum_At_Grid_Edge"] = np.where(
    best["Capacity_kWh"] <= SMALLEST_SIZED, "smallest swept",
    np.where(best["Capacity_kWh"] >= LARGEST_SIZED, "largest swept", ""),
)
# Kept on every row from here on, so no downstream table can quietly compare a
# closed contract against the three that are on sale.
best["Status"] = np.where(best["Tariff"].isin(LEGACY_TARIFFS), "legacy", "selectable")

BEST_COLS = [
    "Label", "Tariff", "Status", "Capacity_kWh", "Optimum_At_Grid_Edge", "Cost_EUR", "Savings_EUR",
    "Net_Annual_EUR", "Net_Annual_StorageOnly_EUR",
    "Capex_EUR", "NPV_EUR", "ROI_pct", "IRR_pct", "Payback_y",
    "Equivalent_Full_Cycles", "Service_Life_y", "Life_Limited_By",
    "Import_kWh", "Export_kWh", "Curtailed_kWh", "Peak_Import_kW",
    # Carried so the read-out can state how much of the winning list's benefit is
    # grid arbitrage rather than self-consumption.
    "Charged_kWh", "Grid_Charged_kWh",
]
best_table = (
    best[BEST_COLS]
    .assign(Tariff=lambda d: pd.Categorical(d["Tariff"], TARIFF_ORDER, ordered=True),
            Label=lambda d: pd.Categorical(d["Label"], LABEL_ORDER, ordered=True))
    .sort_values(["Label", "Tariff"])
    .reset_index(drop=True)
)
print(f"Optimal battery per household and tariff (peak of Net_Annual_EUR) at "
      f"{CAPEX_EUR_PER_KWH:,.0f} EUR/kWh + {CAPEX_FIXED_EUR:,.0f} EUR fixed")
display(best_table.round(2))

In [ ]:
### 8.3 The headline: best selectable tariff per household, and whether the battery pays
# The winner is taken over CURRENT_TARIFF_ORDER. The legacy list is reported in
# its own columns: the size it would want and the net benefit it would reach, so
# a household already on NET metering can see both what it has and what it moves
# to. `Legacy_advantage_EUR` > 0 means the legacy contract is still worth keeping
# while it lasts; it is not evidence for or against buying a battery on a list
# that is no longer for sale.
summary_rows = []
for label in LABEL_ORDER:
    sub = best_table[best_table["Label"] == label].set_index("Tariff")
    current = sub.loc[CURRENT_TARIFF_ORDER]
    win = current.loc[current["Net_Annual_EUR"].idxmax()]
    nb = no_batt.loc[label]
    # Same gap tolerance as 8.1, on the cost the winning size actually reaches.
    with_batt = resolve_winner(current["Cost_EUR"].reindex(CURRENT_TARIFF_ORDER),
                               CURRENT_TARIFF_ORDER)
    row = {
        "Household": label,
        "Best_tariff_no_battery": nb["Cheapest"],
        "Cost_no_battery_EUR": float(nb["Cheapest_EUR"]),
        "Best_tariff_with_battery": win.name,
        "Cheapest_with_battery": with_batt,
        "Optimal_kWh": win["Capacity_kWh"],
        "Cost_EUR": win["Cost_EUR"],
        "Savings_EUR": win["Savings_EUR"],
        "Net_Annual_EUR": win["Net_Annual_EUR"],
        "ROI_pct": win["ROI_pct"],
        "IRR_pct": win["IRR_pct"],
        "Payback_y": win["Payback_y"],
        "Cycles_per_year": win["Equivalent_Full_Cycles"],
        "Battery_pays": "yes" if win["Net_Annual_EUR"] > 0 else "no",
        "Pays_storage_only": "yes" if win["Net_Annual_StorageOnly_EUR"] > 0 else "no",
    }
    for legacy in LEGACY_TARIFFS:
        leg = sub.loc[legacy]
        row[f"{legacy}_kWh"] = leg["Capacity_kWh"]
        row[f"{legacy}_Net_EUR"] = leg["Net_Annual_EUR"]
        # Difference in BATTERY economics, not in the bill. The bill comparison
        # is in 8.1; conflating the two is what makes a legacy contract look
        # like a reason to buy or skip a battery.
        row[f"{legacy}_Net_minus_best_EUR"] = float(
            leg["Net_Annual_EUR"] - win["Net_Annual_EUR"]
        )
    summary_rows.append(row)

headline = pd.DataFrame(summary_rows)
print(f"Best selectable tariff per household  (capex {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh + "
      f"{CAPEX_FIXED_EUR:,.0f} EUR, r = {DISCOUNT_RATE:.0%})")
print(f"Chosen among {', '.join(CURRENT_TARIFF_ORDER)}. "
      f"{', '.join(LEGACY_TARIFFS)} columns are the legacy contract, for holders only.")
display(headline.round(2))

In [ ]:
### 8.4 Optimal battery size against the installed storage cost
# The sizing question, swept over price instead of answered at one price. For
# every level in CAPEX_TABLE_EUR_PER_KWH the net annual benefit is recomputed for
# every (household, tariff, capacity) and the peak is taken, so each cell is that
# household's optimum AT THAT PRICE — not the 250 EUR/kWh optimum re-costed.
#
# The CAPEX_FIXED_EUR install is charged in every scenario. It is what keeps the
# small sizes unprofitable even when storage itself is cheap, and dropping it
# would tilt the whole table towards batteries that only look good because the
# inverter was free.
def optimum_at_cost(capex_per_kwh):
    '''Peak-net-benefit row per (household, tariff) at one installed storage price.'''
    scen = df[df["Capacity_kWh"] > 0].copy()
    scen["Capex"] = scen["Capacity_kWh"] * capex_per_kwh + CAPEX_FIXED_EUR
    scen["Net"] = scen["Savings_EUR"] - scen["Capex"] * scen["Annuity_Factor"]
    top = scen.loc[scen.groupby(["Label", "Tariff"], observed=True)["Net"].idxmax()]
    return top.set_index(["Label", "Tariff"])


optima_by_cost = {c: optimum_at_cost(c) for c in CAPEX_TABLE_EUR_PER_KWH}
INDEX_ALL = pd.MultiIndex.from_product([LABEL_ORDER, TARIFF_ORDER], names=["Label", "Tariff"])
INDEX_CURRENT = pd.MultiIndex.from_product([LABEL_ORDER, CURRENT_TARIFF_ORDER],
                                           names=["Label", "Tariff"])

opt_size = pd.DataFrame(
    {c: t["Capacity_kWh"].reindex(INDEX_ALL) for c, t in optima_by_cost.items()}
)
opt_net = pd.DataFrame(
    {c: t["Net"].reindex(INDEX_ALL) for c, t in optima_by_cost.items()}
)
opt_size.columns.name = opt_net.columns.name = "Storage cost [EUR/kWh]"

### 8.4a Per tariff, across the households
# Three separate tables rather than one stacked frame: a count of households and
# a capacity in kWh do not share a dtype, and stacking them printed "16.00" in
# the count rows.
by_tariff = lambda frame: frame.groupby(level="Tariff").median().reindex(TARIFF_ORDER)
print(f"Optimal battery size [kWh] vs installed storage cost — median over "
      f"{len(LABEL_ORDER)} households, {CAPEX_FIXED_EUR:,.0f} EUR fixed install charged in "
      f"every scenario. Assumed price: {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh.")
print(f"{', '.join(LEGACY_TARIFFS)} is the legacy contract — a reference row, not an option.")
display(by_tariff(opt_size).round(1))

print("\nMedian net annual benefit [EUR/a] at those sizes (negative = no size pays)")
display(by_tariff(opt_net).round(1))

print(f"\nHouseholds with a profitable size, out of {len(LABEL_ORDER)}")
display((opt_net > 0).groupby(level="Tariff").sum().reindex(TARIFF_ORDER).astype(int))

### 8.4b Household by household, on the tariff each one would actually be on
# One row per household: the size to buy at each price, on that household's best
# selectable list from 8.3. This is the table to read a purchase decision off.
best_by_label = dict(zip(headline["Household"], headline["Best_tariff_with_battery"]))
chosen = pd.MultiIndex.from_tuples([(lab, best_by_label[lab]) for lab in LABEL_ORDER])
per_household = opt_size.loc[chosen].copy()
per_household.index = pd.MultiIndex.from_tuples(
    [(lab, best_by_label[lab]) for lab in LABEL_ORDER], names=["Household", "On_tariff"]
)
pays = opt_net.loc[chosen].to_numpy() > 0
print(f"\nOptimal size [kWh] per household at each storage cost, on its best selectable "
      f"tariff. Assumed price is {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh.")
print("A size in brackets is the peak of a curve that never gets above zero — the least-bad "
      "size, not an investment.")
display(
    pd.DataFrame(
        np.where(pays, per_household.map("{:g}".format), "(" + per_household.map("{:g}".format) + ")"),
        index=per_household.index, columns=per_household.columns,
    )
)

print(f"\nNet annual benefit [EUR/a] at those sizes (negative = do not buy)")
display(opt_net.loc[chosen].set_axis(per_household.index).round(1))

### 8.4c Every (household, tariff) pair, sizes only
print(f"\nOptimal size [kWh] for every household on every selectable tariff")
display(opt_size.reindex(INDEX_CURRENT).round(1))

### 8.4d The same question read backwards: what would storage have to cost?
print("\nBreak-even installed cost at each household's optimal size [EUR/kWh] "
      f"(assumed {CAPEX_EUR_PER_KWH:,.0f}):")
display(
    best_table.assign(
        Break_Even=[
            float(df.loc[(df["Label"] == r["Label"]) & (df["Tariff"] == r["Tariff"])
                         & (df["Capacity_kWh"] == r["Capacity_kWh"]),
                         "Break_Even_Capex_EUR_kWh"].iloc[0])
            for _, r in best_table.iterrows()
        ]
    )
    .pivot(index="Label", columns="Tariff", values="Break_Even")
    .reindex(index=LABEL_ORDER, columns=TARIFF_ORDER)
    .round(0)
)

In [ ]:
### 8.5 Cycle count and how the two definitions differ
cycles = (
    sized.pivot_table(index="Capacity_kWh", columns="Tariff",
                      values="Equivalent_Full_Cycles", aggfunc="median")
    .reindex(columns=TARIFF_ORDER)
)
print("Median equivalent full cycles per year, by capacity and tariff")
display(cycles.round(1))

ratio = (
    sized["EFC_AC_Legacy"] / sized["Equivalent_Full_Cycles"].where(sized["Equivalent_Full_Cycles"] > 0)
).dropna()
print(f"\nThe old AC-side formula reports {100 * (1 - ratio.mean()):.1f} % fewer cycles on "
      f"average (exactly the {100 * (1 - DISCHARGE_EFFICIENCY):.0f} % discharge loss).")
print(f"Lifetime cycles at the optimum: {best_table['Equivalent_Full_Cycles'].min():.0f}-"
      f"{best_table['Equivalent_Full_Cycles'].max():.0f} per year, "
      f"service life {best_table['Service_Life_y'].min():.1f}-"
      f"{best_table['Service_Life_y'].max():.1f} y")

## 9. Plots

Three selectable price lists in colour (blue / orange / aqua), always in the same slot order,
with a dash pattern per list as a second, colour-independent channel. The legacy NET-metering
list is drawn in the same grey as every other reference mark in the notebook, thin and dotted:
it is context, not a competitor, and it should not read as the winner of a four-way race.
Every figure has the table above it as its accessible equivalent.

**One panel per household, on its own y scale.** The per-household figures wrap onto a
`PANEL_COLS`-wide grid instead of a single row — sixteen panels side by side left each one
narrower than its own title — and each panel is scaled independently. These households run
from a few hundred EUR/a to well over a thousand, and a shared scale spends the whole axis on
the largest of them while flattening the rest into straight lines. What each panel is for is
the comparison *between the lists inside it*: which curve is lowest, where its peak sits,
which side of zero it is on. For levels compared across households, read the tables in
section 8 or the bar figure, which is a single scale by design.

Layout is handled by one `chart_frame()` call per figure, which measures the header and legend in
inches and gives `tight_layout` what is left. The earlier version placed the title and the
subtitle a fixed *fraction* of the figure height apart, which on a 3.5 in row is less than the
title's own line height — that is what printed them on top of each other — and left the
subtitle unwrapped, which pushed the saved canvas out sideways to reach it.

In [ ]:
### Shared chart styling (validated categorical slots 1-4)
INK, INK_2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#898781", "#fcfcfb"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"

# Colour is spent on the lists a household can actually choose; the legacy list
# is drawn in the same grey as every other reference mark in the notebook, thin
# and dotted. That is the encoding doing the argument: a NET-metering curve that
# sits lowest is a fact about a closed contract, not a recommendation, and it
# should not read as the winner of a four-way race.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
DASHES = ["-", "--", "-."]
TARIFF_COLOR = {t: SERIES[i % len(SERIES)] for i, t in enumerate(CURRENT_TARIFF_ORDER)}
TARIFF_DASH = {t: DASHES[i % len(DASHES)] for i, t in enumerate(CURRENT_TARIFF_ORDER)}
TARIFF_LW = {t: 2.0 for t in CURRENT_TARIFF_ORDER}
TARIFF_LABEL = {t: t for t in CURRENT_TARIFF_ORDER}
for t in LEGACY_TARIFFS:
    TARIFF_COLOR[t], TARIFF_DASH[t], TARIFF_LW[t] = MUTED, (0, (1, 1.4)), 1.5
    TARIFF_LABEL[t] = f"{t} — legacy, reference only"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK_2,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_2, "ytick.color": INK_2, "font.size": 9,
    "legend.frameon": False, "figure.dpi": 110,
})
EUR = FuncFormatter(lambda v, _: f"{v:,.0f}")
PLAIN = FuncFormatter(lambda v, _: f"{v:,.0f}")
KW = FuncFormatter(lambda v, _: f"{v:,.1f}")
PCT = FuncFormatter(lambda v, _: f"{v:+,.0f} %")

# Header and footer geometry in INCHES, not in figure fractions.
#
# The previous version pinned the title at y = 1.000 and the subtitle at
# y = 0.955, so the space between them was 4.5 % of whatever the figure height
# happened to be — 0.16 in on the 3.5 in small-multiple rows, less than the
# 13 pt title itself. Every figure printed the two lines on top of each other.
# Inches are independent of the figure height, so one set of numbers works on a
# 3.4 in row and a 10 in grid alike.
#
# The subtitle is wrapped to the figure width for the same class of reason: as
# one long line it ran off the canvas, and the inline backend saves with
# bbox_inches="tight", which then grew the image sideways to reach it. That is
# what made the marginal-value figure ~18 in wide with the right half empty.
PAD_TOP_IN, TITLE_IN, HEAD_GAP_IN = 0.10, 0.26, 0.08
SUB_LINE_IN, BODY_GAP_IN, LEFT_IN = 0.20, 0.26, 0.16
LEGEND_IN = 0.44
CHAR_IN = 0.062          # mean glyph advance of DejaVu Sans at 9.5 pt


def tariff_handles(kind="line"):
    '''Legend keys that match the marks actually drawn: line + dash pattern for
    the curve figures, a filled patch for the bar figure. A dashed line key next
    to a solid bar advertises a second encoding channel that is not there.'''
    if kind == "bar":
        return [Patch(facecolor=TARIFF_COLOR[t], edgecolor=SURFACE, lw=1.5,
                      label=TARIFF_LABEL[t]) for t in TARIFF_ORDER]
    return [plt.Line2D([], [], color=TARIFF_COLOR[t], ls=TARIFF_DASH[t],
                       lw=TARIFF_LW[t], label=TARIFF_LABEL[t]) for t in TARIFF_ORDER]


def chart_frame(fig, title, subtitle, handles=None, ncol=4):
    '''Reserve the header and legend space, lay the axes out in what is left,
    then fill it. Call once, last, in place of suptitle + legend + tight_layout:
    the space the text needs is measured before `rect` is handed to
    tight_layout, so no two pieces of text can land on each other.

    handles=False draws no legend; a list of handles overrides the tariff keys.
    '''
    w, h = fig.get_figwidth(), fig.get_figheight()
    lines = textwrap.wrap(subtitle, max(40, int((w - 2 * LEFT_IN) / CHAR_IN))) or [""]
    head_in = (PAD_TOP_IN + TITLE_IN + HEAD_GAP_IN
               + SUB_LINE_IN * len(lines) + BODY_GAP_IN)
    foot_in = 0.0 if handles is False else LEGEND_IN

    fig.tight_layout(rect=(0.0, foot_in / h, 1.0, 1.0 - head_in / h))
    fig.text(LEFT_IN / w, 1.0 - PAD_TOP_IN / h, title,
             ha="left", va="top", fontsize=13, fontweight="bold", color=INK)
    fig.text(LEFT_IN / w, 1.0 - (PAD_TOP_IN + TITLE_IN + HEAD_GAP_IN) / h,
             "\n".join(lines), ha="left", va="top", fontsize=9.5,
             color=INK_2, linespacing=1.4)
    if handles is not False:
        fig.legend(handles=handles or tariff_handles(), loc="lower center",
                   bbox_to_anchor=(0.5, 0.22 * LEGEND_IN / h), ncol=ncol, fontsize=9.5)


def edge_label(ax, y, text, side="right"):
    '''One-line annotation pinned to an edge of the axes in AXES coordinates,
    on a surface-coloured pad so it never sits on a curve. Data coordinates are
    the wrong frame for these: the reference lines span the whole axis, and an
    x taken from the data can fall outside the view and be clipped away.'''
    x, ha = (0.995, "right") if side == "right" else (0.005, "left")
    ax.annotate(text, xy=(x, y), xycoords=("axes fraction", "data"),
                xytext=(0, 4), textcoords="offset points", ha=ha, va="bottom",
                color=INK_2, fontsize=8.5, zorder=5,
                bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.6))


PANEL_COLS = 4           # households per row; 16 households -> a 4 x 4 grid


def small_multiples(value, title, subtitle, ylabel, fmt=EUR, zero_line=False,
                    mark_optimum=False, drop_zero=False, sharey=False,
                    baseline_col=None, ncols=PANEL_COLS):
    '''One panel per household, one line per tariff, wrapped onto a grid.

    The row layout this replaced put every household side by side in a single
    3.4 in strip. At five panels that was already tight; at sixteen each panel
    was under an inch wide, which is narrower than its own title, and the figure
    ran off the page sideways. `ncols` wraps instead, so the panel size stays
    the same whatever the household count.

    Every panel carries its OWN y scale (`sharey=False`). These households run
    from a few hundred EUR/a to well over a thousand, and one shared scale
    flattens the small ones into straight lines to make room for the large ones.
    The comparison each panel is for is between tariffs INSIDE it. `sharey=True`
    is kept for the rare figure where the levels themselves are the point, but
    read the panel titles as separate charts, not as one.

    drop_zero    leaves the 0 kWh solve out, for metrics that are 0 there by
                 definition rather than by measurement (cycles) and whose
                 artificial 0 -> peak jump otherwise owns the y range.
    baseline_col draws that household's own no-battery level as a reference.
    '''
    n = len(LABEL_ORDER)
    ncols = max(1, min(int(ncols), n))
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, sharex=True, sharey=sharey,
                             figsize=(3.05 * ncols, 2.5 * nrows), squeeze=False)
    axes = axes.ravel()
    for ax, label in zip(axes, LABEL_ORDER):
        panel = df[df["Label"] == label]
        if baseline_col is not None:
            # Identical across tariffs at 0 kWh — with no store the trajectory
            # is the profile itself — so the median is just a safe reduction.
            ref = float(panel.loc[panel["Capacity_kWh"] == 0.0, baseline_col].median())
            ax.axhline(ref, color=MUTED, lw=1.1, ls=(0, (5, 4)), zorder=1)
        if drop_zero:
            panel = panel[panel["Capacity_kWh"] > 0]
        for tariff in TARIFF_ORDER:
            line = panel[panel["Tariff"] == tariff].sort_values("Capacity_kWh")
            ax.plot(line["Capacity_kWh"], line[value], color=TARIFF_COLOR[tariff],
                    ls=TARIFF_DASH[tariff], lw=TARIFF_LW[tariff],
                    solid_capstyle="round", zorder=2)
            if mark_optimum and len(line):
                sized_line = line[line["Capacity_kWh"] > 0]
                if len(sized_line):
                    top = sized_line.loc[sized_line[value].idxmax()]
                    ax.plot([top["Capacity_kWh"]], [top[value]], marker="o", ms=6,
                            color=TARIFF_COLOR[tariff], mec=SURFACE, mew=1.6, zorder=4)
        if zero_line:
            ax.axhline(0, color=BASELINE, lw=1.0, zorder=1)
        ax.set_title(label, fontsize=9.5)
        ax.yaxis.set_major_formatter(fmt)
        ax.tick_params(labelsize=8)
    for ax in axes[n:]:
        ax.set_visible(False)        # empty slots when n is not a multiple of ncols
    # One y label per row, and x ticks + label on the lowest VISIBLE panel of each
    # column — with independent scales every panel already carries its own ticks,
    # so the words are what would be repetitive, not the numbers. sharex hides
    # tick labels on everything but the last grid row, which is wrong for a
    # column whose last row is one of the blanked-out slots.
    for r in range(nrows):
        axes[r * ncols].set_ylabel(ylabel)
    for i in range(n):
        if i + ncols >= n:
            axes[i].tick_params(labelbottom=True)
            axes[i].set_xlabel("Battery capacity [kWh]", fontsize=9)
    chart_frame(fig, title, subtitle)
    plt.show()


print(f"Style ready — {len(CURRENT_TARIFF_ORDER)} selectable tariffs in colour, "
      f"{len(LEGACY_TARIFFS)} legacy in grey; {len(LABEL_ORDER)} household panels "
      f"on a {math.ceil(len(LABEL_ORDER) / PANEL_COLS)} x {PANEL_COLS} grid.")

In [ ]:
### Annual electricity cost vs battery size
# Independent y per panel, as everywhere in section 9: these households run from
# a ~400 EUR/a bill to a ~1,300 EUR/a one, and a common scale would flatten the
# small ones into straight lines. The comparison this figure is for is between
# tariffs within a panel, not between panels.
small_multiples(
    "Cost_EUR",
    "Perfect-foresight annual electricity cost, by tariff and battery size",
    "Full-year MILP per point, one panel per household on its own scale. NET metering (grey, "
    "dotted) is the legacy contract — settled annually, closed to new customers, shown only as "
    "what its holders pay. Every other list credits exports per interval. Below zero the "
    "household is paid more for its exports than it pays for its imports over the year. "
    "Values in table 8.2.",
    "Annual cost [EUR/a]",
    zero_line=True,
)

In [ ]:
### Net annual benefit — the optimal size is the peak of each curve
small_multiples(
    "Net_Annual_EUR",
    "Net annual benefit = MILP savings - annualized battery cost",
    f"Marker = optimal size on that tariff. Capex {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh + "
    f"{CAPEX_FIXED_EUR:,.0f} EUR fixed, {DISCOUNT_RATE:.0%} discount, service life capped by "
    f"{BATTERY_CYCLE_LIMIT_EFC:,} cycles. Above the zero line = worth buying. Each panel has "
    f"its own scale, so read the sign and the peak position, not the height against the "
    f"neighbours; the levels are in table 8.3.",
    "Net benefit [EUR/a]",
    zero_line=True,
    mark_optimum=True,
)

In [ ]:
### Where the next kWh of storage stops paying for itself
fig, ax = plt.subplots(figsize=(10.5, 4.6))
marginal = (
    df.dropna(subset=["Marginal_Value_EUR_per_kWh"])
    .groupby(["Tariff", "Capacity_kWh"], observed=True)["Marginal_Value_EUR_per_kWh"]
    .median()
)
# Plotted at the midpoint of the interval the difference was taken over.
caps = np.sort(df["Capacity_kWh"].unique())
midpoints = dict(zip(caps[1:], (caps[1:] + caps[:-1]) / 2))

crf_ref = capital_recovery_factor(DISCOUNT_RATE, BATTERY_CALENDAR_LIFE_Y)
# Cost lines first, so the curves and their markers sit on top of them. The
# assumed price gets a solid line; the rest of the scenarios are dashed.
for capex_per_kwh in sorted(set(CAPEX_SCENARIOS_EUR_PER_KWH) | {CAPEX_EUR_PER_KWH}):
    assumed = capex_per_kwh == CAPEX_EUR_PER_KWH
    level = capex_per_kwh * crf_ref
    ax.axhline(level, color=INK_2 if assumed else MUTED, lw=1.2 if assumed else 1.1,
               ls="-" if assumed else (0, (5, 4)), zorder=1)
    edge_label(ax, level, f"{capex_per_kwh:,.0f} EUR/kWh installed  ({level:,.1f} EUR/kWh/a)"
                          + ("  <- assumed" if assumed else ""))

for tariff in TARIFF_ORDER:
    series = marginal.loc[tariff]
    ax.plot([midpoints[c] for c in series.index], series.values,
            color=TARIFF_COLOR[tariff], ls=TARIFF_DASH[tariff], lw=TARIFF_LW[tariff],
            marker="o", ms=5, mfc=SURFACE, mew=1.4, zorder=3)

ax.set_ylim(bottom=0)
ax.margins(x=0.02)
ax.set_xlabel("Battery capacity [kWh]")
ax.set_ylabel("Marginal value of capacity [EUR/kWh/a]")
ax.yaxis.set_major_formatter(PLAIN)
chart_frame(fig, "Where the next kWh of storage stops paying for itself",
      f"Median over the {len(LABEL_ORDER)} households. Capacity is worth adding while its "
      f"curve sits above a cost line. Those lines are the per-kWh storage cost annualized "
      f"over the full {BATTERY_CALENDAR_LIFE_Y} y calendar life; they exclude the "
      f"{CAPEX_FIXED_EUR:,.0f} EUR install, which is charged once regardless of size. Where a "
      f"curve crosses the {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh line is the size in table 8.4.")
plt.show()

In [ ]:
### Equivalent full cycles per year
# drop_zero: EFC is 0 at 0 kWh by definition, not by measurement. Including it
# drew a vertical 0 -> 600 spike into the first swept size in every panel, and
# that artefact set the y range the rest of the curve then had to live in.
small_multiples(
    "Equivalent_Full_Cycles",
    "How hard each tariff works the battery",
    f"Equivalent full cycles a year on stored energy, (in + out) / (2 x capacity). "
    f"A pack rated {BATTERY_CYCLE_LIMIT_EFC:,} EFC is spent in "
    f"{BATTERY_CYCLE_LIMIT_EFC // BATTERY_CALENDAR_LIFE_Y} cycles/a, so a curve above that "
    f"level is buying its savings out of the pack's service life. Own scale per household; "
    f"medians across all of them are in table 8.5.",
    "Equivalent full cycles [1/a]",
    fmt=PLAIN,
    drop_zero=True,
)

In [ ]:
### The headline comparison: net annual benefit at each household's optimal size
# Horizontal bars, one row per household. Vertical grouped bars put N x 4 bars
# and N household names on a shared x axis; past about six households the names
# collide and the bars are a couple of points wide. Rotating the labels only
# trades one unreadable axis for another. Turned on its side, the household name
# is a full-length horizontal label whatever the count, and the figure grows
# downwards — which a notebook scrolls anyway.
n = len(LABEL_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 0.44 * n + 2.2), sharey=True)
y = np.arange(n)
height = 0.78 / len(TARIFF_ORDER)

for ax, (value, xlabel, fmt, panel_title) in zip(axes, [
    ("Net_Annual_EUR", "Net benefit [EUR/a]", EUR, "Net annual benefit at the optimal size"),
    ("ROI_pct", "Discounted ROI [%]", PLAIN, "Return on the capital, over the service life"),
]):
    for i, tariff in enumerate(TARIFF_ORDER):
        vals = [
            float(best_table.loc[(best_table["Label"] == lab)
                                 & (best_table["Tariff"] == tariff), value].iloc[0])
            for lab in LABEL_ORDER
        ]
        # Offsets run downwards so the bar order inside a group matches the
        # legend order top to bottom (the y axis is inverted below).
        ax.barh(y + (i - (len(TARIFF_ORDER) - 1) / 2) * height, vals, height * 0.9,
                color=TARIFF_COLOR[tariff], zorder=2)
    ax.axvline(0, color=BASELINE, lw=1.0, zorder=3)
    ax.set_xlabel(xlabel)
    ax.set_title(panel_title)
    ax.xaxis.set_major_formatter(fmt)
    ax.margins(x=0.08)
    ax.grid(axis="y", visible=False)

axes[0].set_yticks(y, LABEL_ORDER, fontsize=8.5)
axes[0].set_ylim(n - 0.5, -0.5)          # first household at the top
chart_frame(fig, "Which tariff makes a battery worth buying",
      f"Each household at its own best size on each tariff. Bars left of zero lose money at "
      f"{CAPEX_EUR_PER_KWH:,.0f} EUR/kWh + {CAPEX_FIXED_EUR:,.0f} EUR. The grey bar is the "
      f"legacy NET-metering contract, for households that already hold it — it is not one of "
      f"the three lists on offer. Values in table 8.2.",
      handles=tariff_handles("bar"))
plt.show()

In [ ]:
### What the battery actually does: throughput and curtailment
# Peak import used to be the third panel here; it is the whole of the next
# figure now, where there is room for the no-battery and contracted-power
# references that make a kW number mean anything.
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
grouped = df[df["Capacity_kWh"] > 0].groupby(["Tariff", "Capacity_kWh"], observed=True)

for ax, (value, title, ylabel) in zip(axes, [
    ("Grid_Charged_kWh", "Charged from the grid", "Energy [kWh/a]"),
    ("Curtailed_kWh", "PV curtailed", "Energy [kWh/a]"),
]):
    for tariff in TARIFF_ORDER:
        series = grouped[value].median().loc[tariff]
        ax.plot(series.index, series.values, color=TARIFF_COLOR[tariff],
                ls=TARIFF_DASH[tariff], lw=TARIFF_LW[tariff])
    ax.set_title(title)
    ax.set_xlabel("Battery capacity [kWh]")
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_formatter(PLAIN)

chart_frame(fig, "Where each tariff's savings come from",
      f"Median across the {len(LABEL_ORDER)} households. Grid charging is trading revenue "
      f"rather than self-consumption; curtailment is the model refusing to export at a loss. "
      f"Series that sit on zero are drawn over each other — read them off the dash pattern.")
plt.show()

In [ ]:
### What the battery does to the peak grid import
# The cost curves say what the battery is worth; this says what it does to the
# connection. Peak import is billed against contracted power under the SI network
# tariff, and the block-arbitrage lists charge from the grid at the cheapest hour
# of the day — a load the house never had — so their peak climbs with the
# inverter rating instead of falling with self-consumption.
peak_base = (
    df.loc[df["Capacity_kWh"] == 0.0]
    .set_index(["Label", "Tariff"])["Peak_Import_kW"]
)
peaks = df[df["Capacity_kWh"] > 0].copy()
peaks["Peak_No_Battery_kW"] = [
    peak_base.loc[(lab, t)] for lab, t in zip(peaks["Label"], peaks["Tariff"])
]
peaks["Peak_Change_pct"] = 100.0 * (
    peaks["Peak_Import_kW"] / peaks["Peak_No_Battery_kW"] - 1.0
)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.2))
by_cap = peaks.groupby(["Tariff", "Capacity_kWh"], observed=True)

for tariff in TARIFF_ORDER:
    abs_kw = by_cap["Peak_Import_kW"].median().loc[tariff]
    rel = by_cap["Peak_Change_pct"].median().loc[tariff]
    lo = by_cap["Peak_Change_pct"].min().loc[tariff]
    hi = by_cap["Peak_Change_pct"].max().loc[tariff]
    axes[0].plot(abs_kw.index, abs_kw.values, color=TARIFF_COLOR[tariff],
                 ls=TARIFF_DASH[tariff], lw=TARIFF_LW[tariff], zorder=3)
    # Band = the households' spread, so a median cannot hide one runaway.
    axes[1].fill_between(rel.index, lo.values, hi.values, color=TARIFF_COLOR[tariff],
                         alpha=0.10, lw=0, zorder=1)
    axes[1].plot(rel.index, rel.values, color=TARIFF_COLOR[tariff],
                 ls=TARIFF_DASH[tariff], lw=TARIFF_LW[tariff], zorder=3)

base_kw = float(peak_base.groupby(level="Label").median().median())
contracted_kw = float(df.groupby("Label")["Contracted_Power_kW"].first().median())
axes[0].axhline(base_kw, color=BASELINE, lw=1.2, zorder=2)
edge_label(axes[0], base_kw, f"no battery: {base_kw:,.1f} kW", side="left")
axes[0].axhline(contracted_kw, color=MUTED, lw=1.1, ls=(0, (5, 4)), zorder=2)
edge_label(axes[0], contracted_kw, f"median contracted power: {contracted_kw:,.1f} kW")
axes[0].set_title(f"Peak grid import, median of the {len(LABEL_ORDER)} households")
axes[0].set_ylabel("Peak import [kW]")
axes[0].set_ylim(bottom=0)
axes[0].yaxis.set_major_formatter(KW)

axes[1].axhline(0, color=BASELINE, lw=1.2, zorder=2)
axes[1].set_title("Change against the same household with no battery")
axes[1].set_ylabel("Change in peak import [%]")
axes[1].yaxis.set_major_formatter(PCT)

for ax in axes:
    ax.set_xlabel("Battery capacity [kWh]")
    ax.margins(x=0.02)

chart_frame(fig, "What the battery does to the peak grid import",
      f"Left: median peak, against the no-battery peak and the median contracted power "
      f"({contracted_kw:,.1f} kW) the network tariff bills against. Right: change against each "
      f"household's own no-battery peak — line is the median, band is the full spread of all "
      f"{len(LABEL_ORDER)}. Above zero the battery has made the connection peak worse, not "
      f"better.")
plt.show()

### The same thing household by household, each on its own scale
small_multiples(
    "Peak_Import_kW",
    "Peak grid import, household by household",
    f"Dashed line is that household's own no-battery peak; each panel is on its own kW scale, "
    f"so read every curve against the dashed line in the SAME panel, not across panels. Power "
    f"is capped at min({C_RATE:g} x capacity, {INVERTER_MAX_KW:g} kW), which is the ceiling "
    f"the grid-charging tariffs run into from about {INVERTER_MAX_KW / C_RATE:g} kWh up.",
    "Peak import [kW]",
    fmt=KW,
    drop_zero=True,
    baseline_col="Peak_Import_kW",
)

### Aggregate and extremes, since a median peak is not what gets billed
worst_row = peaks.loc[peaks["Peak_Change_pct"].idxmax()]
best_row = peaks.loc[peaks["Peak_Change_pct"].idxmin()]
print(f"Peak grid import across {len(LABEL_ORDER)} households x {len(TARIFF_ORDER)} tariffs "
      f"x {len(BATTERY_SIZES_KWH) - 1} sized runs")
print(f"  no battery, summed over the {len(LABEL_ORDER)} households: "
      f"{peak_base.groupby(level='Label').median().sum():,.1f} kW")
for tariff in TARIFF_ORDER:
    sub = peaks[peaks["Tariff"] == tariff]
    at_opt = best_table[best_table["Tariff"] == tariff]
    tag = "  (legacy)" if tariff in LEGACY_TARIFFS else ""
    print(f"  {tariff:>12}: median {sub['Peak_Change_pct'].median():+6.1f} % over all sizes, "
          f"{at_opt['Peak_Import_kW'].sum():6.1f} kW summed at the optimal sizes, "
          f"worst single run {sub['Peak_Change_pct'].max():+6.1f} %{tag}")
print(f"  worst: {worst_row['Label']} on {worst_row['Tariff']} at {worst_row['Capacity_kWh']:g} kWh — "
      f"{worst_row['Peak_No_Battery_kW']:.1f} -> {worst_row['Peak_Import_kW']:.1f} kW "
      f"({worst_row['Peak_Change_pct']:+.1f} %), {worst_row['Grid_Charged_kWh']:,.0f} kWh/a of it "
      f"charged from the grid")
print(f"  best:  {best_row['Label']} on {best_row['Tariff']} at {best_row['Capacity_kWh']:g} kWh — "
      f"{best_row['Peak_No_Battery_kW']:.1f} -> {best_row['Peak_Import_kW']:.1f} kW "
      f"({best_row['Peak_Change_pct']:+.1f} %)")

## 10. Read-out

In [ ]:
lines = [
    f"{len(LABEL_ORDER)} PV households from {', '.join(PV_DATASETS)}, calendar year 2024",
    f"{N_STEPS_FULL_YEAR:,} intervals of {step_minutes:.0f} min each — the whole year, every run",
    f"{len(TARIFF_ORDER)} GEN-I price lists x {len(BATTERY_SIZES_KWH)} capacities "
    f"({BATTERY_SIZES_KWH.min():.0f}-{BATTERY_SIZES_KWH.max():.0f} kWh) = {len(df)} full-year MILPs",
    f"Selectable: {', '.join(CURRENT_TARIFF_ORDER)}.  Legacy, reference only: "
    f"{', '.join(LEGACY_TARIFFS)} — closed to new contracts, so it never wins anything below",
    f"Storage at {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh + {CAPEX_FIXED_EUR:,.0f} EUR fixed, "
    f"r = {DISCOUNT_RATE:.0%}",
    f"SI tariff regime {PRICING_REFERENCE_YEAR}, peak reset {PEAK_RESET_MONTHS or 'never (annual ratchet)'}",
    "",
    "WITHOUT A BATTERY",
]
for label in LABEL_ORDER:
    r = no_batt.loc[label]
    legacy_bits = "  ".join(
        f"[{leg} {float(r[leg]):,.2f}, {float(r[f'{leg} - best_EUR']):+,.2f} vs best]"
        for leg in LEGACY_TARIFFS
    )
    lines.append(
        f"  {label:>16}: cheapest {str(r['Cheapest']):<24} "
        f"{float(r['Cheapest_EUR']):8,.2f} EUR/a"
        f"   (spread across the {len(CURRENT_TARIFF_ORDER)} selectable lists "
        f"{float(r['Spread_EUR']):7,.2f} EUR/a)  {legacy_bits}"
    )

lines += ["", "WITH THE BEST BATTERY THE BEST SELECTABLE TARIFF SUPPORTS"]
for _, r in headline.iterrows():
    lines.append(
        f"  {r['Household']:>16}: {str(r['Best_tariff_with_battery']):<12} "
        f"{r['Optimal_kWh']:5.1f} kWh -> saves {r['Savings_EUR']:8,.2f} EUR/a, "
        f"net {r['Net_Annual_EUR']:+8,.2f} EUR/a, ROI {r['ROI_pct']:+7.1f} %, "
        f"payback {r['Payback_y']:5.1f} y, {r['Cycles_per_year']:5.0f} cycles/a "
        f"[pays: {r['Battery_pays']}; storage cost only: {r['Pays_storage_only']}]"
    )

n_pays = int((headline["Battery_pays"] == "yes").sum())
n_storage = int((headline["Pays_storage_only"] == "yes").sum())
edge = best_table[best_table["Optimum_At_Grid_Edge"] != ""]
lines += [
    "",
    f"  -> {n_pays} of {len(headline)} households have ANY capacity with a positive net annual "
    f"benefit at {CAPEX_EUR_PER_KWH:,.0f} EUR/kWh + {CAPEX_FIXED_EUR:,.0f} EUR fixed.",
    f"  -> {n_storage} of {len(headline)} do once the {CAPEX_FIXED_EUR:,.0f} EUR install is "
    f"treated as sunk and only the storage is charged for — that fixed term, not the",
    f"     per-kWh price, is what decides it at these sizes.",
]
if len(edge):
    lines.append(
        f"  -> {len(edge)} of {len(best_table)} (household, tariff) optima sit on the edge of the "
        f"swept grid ({SMALLEST_SIZED:g}-{LARGEST_SIZED:g} kWh), so the grid rather than the"
    )
    lines.append("     economics is picking those sizes — read them as 'as small as swept', not as an optimum.")

lines += ["", "THE LEGACY CONTRACT — for households already on NET metering, not a baseline"]
for legacy in LEGACY_TARIFFS:
    cost_delta = no_batt[f"{legacy} - best_EUR"]       # negative = legacy bill is lower
    leg_rows = best_table[best_table["Tariff"] == legacy]
    pad = " " * (len(legacy) + 2)
    lines.append(
        f"  {legacy}: THE BILL — cheaper than the best selectable list for "
        f"{int((cost_delta < 0).sum())} of {len(cost_delta)} households with no battery, by "
        f"{-cost_delta.median():,.2f} EUR/a at the median"
    )
    lines.append(
        f"{pad}(worst case for the holder {-cost_delta.max():,.2f}, best "
        f"{-cost_delta.min():,.2f}). That is what ending the contract costs them."
    )
    lines.append(
        f"{pad}THE BATTERY — worth {leg_rows['Savings_EUR'].median():,.2f} EUR/a at the median "
        f"optimum of {leg_rows['Capacity_kWh'].median():.1f} kWh, against "
        f"{headline['Savings_EUR'].median():,.2f} EUR/a on the best"
    )
    lines.append(
        f"{pad}selectable list. Annual netting already credits exported energy at the same "
        f"supplier price the household buys at, so a battery has no"
    )
    lines.append(
        f"{pad}energy spread left to capture on this list — only the network charge and the "
        f"peak, which is close to nothing here. Storage and NET"
    )
    lines.append(
        f"{pad}metering are substitutes, not complements: while the contract lasts it is doing "
        f"the battery's job. Size the pack for the selectable"
    )
    lines.append(
        f"{pad}list the household falls back to when the contract ends — that is the row above, "
        f"not this one."
    )

lines += ["", "EXTREMES — where the assumptions show"]
hi = headline.loc[headline["Net_Annual_EUR"].idxmax()]
lo = headline.loc[headline["Net_Annual_EUR"].idxmin()]
for tag, r in (("best ", hi), ("worst", lo)):
    key = [k for k in HOUSEHOLD_KEYS if HOUSEHOLD_LABELS[k] == r["Household"]][0]
    prof = selected[(selected["Dataset"] == key[0]) & (selected["Household"] == key[1])].iloc[0]
    run = df[(df["Label"] == r["Household"]) & (df["Tariff"] == r["Best_tariff_with_battery"])
             & (df["Capacity_kWh"] == r["Optimal_kWh"])].iloc[0]
    lines.append(
        f"  {tag} case {r['Household']:>16} on {str(r['Best_tariff_with_battery']):<12} "
        f"net {r['Net_Annual_EUR']:+8,.2f} EUR/a"
    )
    lines.append(
        f"          load {prof['Consumption_kWh']:6,.0f} kWh/a, PV {prof['Generation_kWh']:6,.0f} kWh/a "
        f"(PV/load {prof['PV_to_Load']:.2f}), {prof['Role']}"
    )
    lines.append(
        f"          at the optimum: imports {run['Import_kWh']:6,.0f}, exports {run['Export_kWh']:6,.0f}, "
        f"curtails {run['Curtailed_kWh']:6,.0f} kWh/a; {run['Grid_Charged_kWh']:6,.0f} kWh/a of the "
        f"charging came from the grid ({100 * run['Grid_Charged_kWh'] / max(run['Charged_kWh'], 1e-9):.0f} % "
        f"of throughput — that share is arbitrage, not self-consumption)"
    )
    lines.append(
        f"          peak import {run['Peak_Import_kW']:5.2f} kW against a contracted "
        f"{run['Contracted_Power_kW']:5.2f} kW"
    )

lines += ["", f"PER TARIFF, MEDIAN OVER THE {len(LABEL_ORDER)} HOUSEHOLDS AT THEIR OPTIMAL SIZE"]
for tariff in TARIFF_ORDER:
    sub = best_table[best_table["Tariff"] == tariff]
    tag = "  (legacy)" if tariff in LEGACY_TARIFFS else ""
    lines.append(
        f"  {tariff:>12}: {sub['Capacity_kWh'].median():5.1f} kWh, "
        f"saves {sub['Savings_EUR'].median():8,.2f} EUR/a, "
        f"net {sub['Net_Annual_EUR'].median():+8,.2f} EUR/a, "
        f"ROI {sub['ROI_pct'].median():+7.1f} %, "
        f"{sub['Equivalent_Full_Cycles'].median():5.0f} cycles/a, "
        f"life {sub['Service_Life_y'].median():4.1f} y "
        f"({sub['Life_Limited_By'].mode().iloc[0]}-limited){tag}"
    )
# Where the money comes from, per list. A large grid-charged share means the
# saving is trading revenue that depends on the 2024 price spread repeating —
# not on self-consumption, and not on anything the household controls.
lines.append("")
for tariff in TARIFF_ORDER:
    sub = best_table[best_table["Tariff"] == tariff]
    share = 100.0 * sub["Grid_Charged_kWh"] / sub["Charged_kWh"].where(sub["Charged_kWh"] > 0)
    lines.append(
        f"  {tariff:>12}: {share.median():5.1f} % of the charge energy at the optimal size came "
        f"from the grid, not from PV (range {share.min():.0f}-{share.max():.0f} %)"
    )

lines += ["", "OPTIMAL SIZE AGAINST THE STORAGE PRICE — median over the households, per tariff"]
lines.append(f"  {'EUR/kWh':>12} " + "".join(f"{c:>7,.0f}" for c in CAPEX_TABLE_EUR_PER_KWH))
for tariff in TARIFF_ORDER:
    med = opt_size.xs(tariff, level="Tariff").median()
    tag = "  (legacy)" if tariff in LEGACY_TARIFFS else ""
    lines.append(f"  {tariff:>12} " + "".join(f"{v:>7.1f}" for v in med) + f"  kWh{tag}")
for tariff in TARIFF_ORDER:
    n_ok = (opt_net.xs(tariff, level="Tariff") > 0).sum()
    tag = "  (legacy)" if tariff in LEGACY_TARIFFS else ""
    lines.append(f"  {tariff:>12} " + "".join(f"{v:>7d}" for v in n_ok)
                 + f"  households in profit of {len(LABEL_ORDER)}{tag}")

lines += ["", "SENSITIVITY — break-even installed cost at each household's optimum"]
for _, r in best_table.iterrows():
    if str(r["Tariff"]) == str(
        headline.loc[headline["Household"] == r["Label"], "Best_tariff_with_battery"].iloc[0]
    ):
        lines.append(
            f"  {r['Label']:>16} on {r['Tariff']:<12}: pays for itself below "
            f"{float(df.loc[(df['Label'] == r['Label']) & (df['Tariff'] == r['Tariff']) & (df['Capacity_kWh'] == r['Capacity_kWh']), 'Break_Even_Capex_EUR_kWh'].iloc[0]):7,.0f} EUR/kWh"
            f"   (assumed {CAPEX_EUR_PER_KWH:,.0f})"
        )

print("\n".join(lines))

### Caveats worth keeping in view

* **Perfect foresight.** Every point is an upper bound; a real controller (DQN, MPC) captures
  only part of it, so the true optimal size is at or below what these curves show, and every
  ROI here is optimistic.
* **0.1 % MIP gap.** Each point is within 0.1 % of its own optimum. That is a few tenths of a
  euro on these bills, well under the spacing between adjacent capacities. Set
  `SOLVER_GAP_REL = 0` for proven optima at 10–20x the runtime.
* **NET metering is a legacy contract.** It is closed to new customers, so it is
  excluded from every winner and every headline in section 8 and reported as a grey
  reference line instead. It is still the right number for a household that holds the
  contract — but the battery that household buys will outlive it, so size the pack on
  the selectable list it falls back to, not on this one.
* **NET metering is modelled as supplier-energy netting only.** Network charges and levies are
  billed on gross metered offtake, which is the post-2024 network act. The list's extra
  courtesy — one monthly fee waived per whole MWh of *surplus* beyond annual consumption — is
  not modelled; it is worth at most a couple of euro a year and only for households that
  export more than they import.
* **Arbitrage through the grid — including on the list that wins.** The model may buy
  cheap grid energy and sell it back later. On the dynamic list that is the SIPX spread;
  on Aktivni it is the fixed block spread, 0.04090 in and 0.14990 out, available every
  single day and bounded only by the inverter. Around half the charge energy at the
  optimal sizes comes from the grid on both of them (the read-out prints the share per
  list), so a large part of what makes them the best lists here is trading revenue, not
  PV self-consumption. Before acting on either, check whether the supplier permits
  grid charging for arbitrage at all. The *Charged from the grid* panel makes the scale
  of it visible.
  Where that line is a large share of throughput, the savings are trading revenue that depends
  on the 2024 SIPX spread repeating, not on PV self-consumption. It is also what drives the
  high cycle counts, and therefore the cycle-limited service life.
* **Peak charges.** `PEAK_RESET_MONTHS = None` ratchets one peak across the whole year and
  charges the excess once. The real SI regulation settles excess power per calendar month;
  `PEAK_RESET_MONTHS = 1` switches to that and makes peak shaving a larger share of the
  battery's value — identically for all four tariffs, since they share a network tariff.
* **Contracted power.** `CONTRACTED_POWER_KW = None` uses the environment's default, the
  household's own historical no-battery peak divided by 1.5, so every household starts out of
  contract and pays an excess-power charge. That inflates the battery's value uniformly across
  the four tariffs; set a real connection size to remove it.
* **The Fluvius clock is 1–2 h off (section 4).** The input timestamps are Brussels local time
  labelled `Z`, so every interval is priced under a tariff block 1–2 hours later than it
  really occurred, and the SMP series is aligned to it with the same shift. All four tariffs
  read the same shifted clock, so the *comparison* holds; the absolute bills, and in
  particular anything that hinges on block boundaries (the Aktivni list, the ratchet peak),
  carry the error. Repository-wide input fix, not a notebook fix.
* **`Feed_In_Volume_kWh` is metered feed-in, not gross PV.** The repository treats it as the
  generation series throughout, and roughly 12 % of intervals have both registers nonzero, so
  it is not a purely netted register — but self-consumption that never crossed the meter is
  not in this data, and the PV-to-load ratios above should be read with that in mind.
* **One price year, sixteen households.** 2024 SMP volatility drives most of the dynamic
  list's advantage. Re-run against another price year before generalizing, and remember
  sixteen households is a sample, not a distribution.